# Task 5: Sales Prediction Using Python
Objective: Build a regression model that predicts product sales based on advertising spend across different media channels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

### Download and Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv"
urllib.request.urlretrieve(url, "Advertising.csv")

df = pd.read_csv("Advertising.csv", index_col=0)
df.head()

### Exploratory Data Analysis (EDA)

In [ ]:
print("Null Check:\n", df.isnull().sum())
print("\nDescriptive Statistics:")
display(df.describe())

In [ ]:
# Pairplot of all features
sns.pairplot(df)
plt.suptitle("Pairplot of Advertising Features", y=1.02)
plt.show()

### Individual Scatter Plots

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(data=df, x="TV", y="Sales", ax=axs[0])
axs[0].set_title("Sales vs TV Spend")

sns.scatterplot(data=df, x="Radio", y="Sales", ax=axs[1])
axs[1].set_title("Sales vs Radio Spend")

sns.scatterplot(data=df, x="Newspaper", y="Sales", ax=axs[2])
axs[2].set_title("Sales vs Newspaper Spend")

plt.tight_layout()
plt.show()

### Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

### Train/Test Split

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Train Models
Training a baseline Linear Regression and a Random Forest Regressor.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

### Evaluation

In [ ]:
def evaluate_regressor(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"R2:   {r2_score(y_true, y_pred):.4f}\n")

evaluate_regressor("Linear Regression", y_test, lr_pred)
evaluate_regressor("Random Forest Regressor", y_test, rf_pred)

### Residual Plot for the Best Model
The Random Forest Regressor performed significantly better based on the R2 score.

In [ ]:
residuals = y_test - rf_pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=rf_pred, y=residuals)
plt.axhline(y=0, color="r", linestyle="--")
plt.xlabel("Predicted Sales")
plt.ylabel("Residuals")
plt.title("Residual Plot (Random Forest)")
plt.show()

The residuals appear somewhat randomly distributed around the zero line, indicating that the model is capturing most of the systematic variance.

### Interpretation
Which advertising channel has the highest impact on sales?

In [ ]:
# Using feature importances from Random Forest
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.plot(kind="barh", color=["blue", "orange", "green"])
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.show()

# Using coefficients from Linear Regression
coeffs = pd.Series(lr.coef_, index=X.columns)
print("Linear Regression Coefficients:")
print(coeffs)